# 🧬 Mapping Molecular Landscapes
## Open-Source Approaches to Chemical Space Exploration

**Workshop | 65-minute hands-on session**
**Datasets:** MMV Malaria Box (400 anti-malarial compounds) · AfroDb African Natural Products (903 compounds) · PubChem AID 2302 (2 000-compound screen, Extension 3)

---

### Learning Objectives
By the end of this notebook you will be able to:
1. Load a real molecular dataset and apply the standard preprocessing pipeline: parse SMILES, remove salts, and extract the largest organic moiety
2. Represent molecules numerically — compute Lipinski physicochemical descriptors (MW, LogP, HBD, HBA, TPSA), apply the Rule of 5, and generate Morgan (ECFP4) fingerprints
3. Measure molecular similarity using the Tanimoto coefficient and interpret block-structured similarity heatmaps
4. Reduce high-dimensional chemical space to 2D and 3D using PCA and UMAP, and explain when each method is appropriate
5. Interpret chemical-space maps by colouring them by biological activity, physicochemical properties, and Murcko scaffold diversity
6. Contextualise results biologically — identify activity cliffs, discuss SAR implications, and compare the chemical space of African natural products vs. synthetic anti-malarials

---

### How to Use This Notebook

- Cells marked **** are **fill-in-the-blank** — you write the key logic.
- All boilerplate (imports, plotting, helper functions) is **pre-written** — just run those cells.


---
## Part A — Environment Setup & Imports ⏱ *~3 min (run, don't modify)*

> 📌 **Instructor note:** Run this cell first and confirm everyone sees `✅ All dependencies loaded.`  
> Fix any import errors here — they will cascade throughout the notebook if left unresolved.  
> If RDKit is missing, check that you activated the `cheminf` conda environment.


In [ ]:

# ============================================================
# CELL A1 — Imports & Constants
# PRE-WRITTEN: run this cell first, do not modify
# ============================================================
import sys, os

# Make src/ importable — chem_utils.py (fragment_count, numpy_to_rdkit_fp) lives there.
# Note: __file__ is not defined in Jupyter; we navigate relative to the notebook's cwd.
_notebook_dir = os.path.abspath(os.getcwd())          # .../caismd_2026_chemspace_xplr/notebooks
_src_dir       = os.path.join(_notebook_dir, "..", "src")
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import seaborn as sns
from tqdm import tqdm

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdMolDescriptors, rdmolops, AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import Draw
from rdkit.ML.Cluster import Butina

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap

# Workshop utilities (format conversions and correctness checks) —
# these live in src/chem_utils.py to keep the notebook focused on concepts.
from chem_utils import fragment_count, numpy_to_rdkit_fp

# ── Global constants ────────────────────────────────────────
RANDOM_SEED = 42          # reproducibility — pass to all stochastic calls

FP_RADIUS   = 2           # Morgan radius (2 = ECFP4, the cheminformatics standard)
FP_NBITS    = 2048        # fingerprint bit-vector length

# Colour palette: consistent across all plots in this notebook
PALETTE = {
    'MMV'    : '#2196F3',   # blue   — MMV Malaria Box
    'AfroDb' : '#4CAF50',   # green  — AfroDb African natural products
}

print("✅ All libraries loaded successfully.")


---
## Part B — Data Loading & Molecule Standardisation ⏱ *~8 min*

> 📌 **What happened before this notebook:**
> `src/prepare_datasets.py` already validated all SMILES and canonicalised them.
> The CSVs you are loading contain no invalid SMILES.

> 📌 **Why do we still standardise here?**
> In real projects, datasets arrive with salts (`"CC(=O)O.[Na+]"`), mixtures (`"c1ccccc1.O"`),
> and occasionally unparseable strings. The two-step pattern — **(1) parse → (2) largest fragment** —
> is the standard opening of every cheminformatics pipeline.

### 🧂 Salt removal

Many compounds are stored as **salts**: the active molecule paired with a counter-ion, joined by a `.` in SMILES:

| Raw SMILES | What we want |
|---|---|
| `CC(=O)[O-].[Na+]` | Acetate `CC(=O)[O-]` |
| `CC(C)(C)NCC(O)c1ccc(Cl)cc1.Cl` | Tulobuterol free base |

We keep the **largest organic fragment** using `LargestFragmentChooser(preferOrganic=True)`.  
"Organic" = contains at least one carbon atom. Without this, a large inorganic counter-ion could be kept by mistake.

### The Dataset
We are working with a combined CSV containing:
- **MMV Malaria Box** — 400 anti-malarial compounds (confirmed activity vs. *P. falciparum*)
- **AfroDb** — 903 unique African natural products (Ntie-Kang *et al.*, *PLoS ONE* 2013)

Key columns:
| Column | Description |
|---|---|
| `SMILES` | RDKit canonical SMILES (pre-validated) |
| `COMPOUND_ID` | Unique identifier |
| `source` | `'MMV'` or `'AfroDb'` |
| `pIC50` | −log₁₀(IC₅₀ in M) for anti-malarial activity (MMV only; NaN for AfroDb) |
| `MW`, `LogP`, `HBD`, `HBA`, `TPSA`, `RotBonds` | Precomputed RDKit descriptors |


In [ ]:

# ============================================================
# CELL B1 — Load the Dataset
# PRE-WRITTEN: just run this cell
# ============================================================

# The combined CSV is generated from the raw data files shipped with the repo.
# If the file is missing, run the preparation script first:
#
#   python src/prepare_datasets.py
#
# This reads:
#   data/MalariaBox400compoundsDec2014.xls  →  data/malaria_box.csv         (400 compounds)
#   data/AfroDB_3D.sdf                      →  data/afrodb_subset.csv       (903 compounds)
# and merges them into:
#   data/malaria_box_afrodb_combined.csv    →  used below
#
# See data/README_data.md for full column documentation.

DATA_PATH = "../data/malaria_box_afrodb_combined.csv"

try:
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Loaded: {DATA_PATH}")
except FileNotFoundError:
    print("❌ Combined data file not found.")
    print("   Please run the preparation script from the workshop root:")
    print()
    print("       python src/prepare_datasets.py")
    print()
    print("   Then re-run this cell.")
    raise

print(f"\nDataset shape : {df.shape}")
print(f"Source counts :\n{df['source'].value_counts()}")
print()
display(df.head(5))


In [ ]:

# ============================================================
# CELL B2 — SMILES Parsing & Molecule Standardisation
# PRE-WRITTEN helpers + 2 TODOs
# ============================================================
#
# Even though the CSV SMILES are canonical, real datasets contain:
#   • Salts:    "CC(=O)O.[Na+]"  → you want the drug, not the counter-ion
#   • Mixtures: "c1ccccc1.O"     → the largest fragment is the molecule of interest
#   • Empty strings or NaN       → must return None
#
# Standard workflow:
#   1. Parse SMILES → Mol object            (TODO 1)
#   2. Remove salts / keep largest fragment (TODO 2)
#   3. Drop rows where mol is still None
#
# Note: steps 1–2 are idempotent for clean SMILES — safe to always run.

from rdkit.Chem.MolStandardize import rdMolStandardize

# --- PRE-WRITTEN: helpers (do not modify) ---

def smiles_to_mol(smi):
    """Parse a SMILES string to an RDKit Mol. Returns None if invalid."""
    if not isinstance(smi, str) or not smi.strip():
        return None
    from rdkit import RDLogger
    RDLogger.DisableLog('rdApp.*')
    mol = Chem.MolFromSmiles(smi.strip())
    RDLogger.EnableLog('rdApp.*')
    return mol


from rdkit import RDLogger as _RDLogger
_RDLogger.DisableLog('rdApp.*')
_lfc = rdMolStandardize.LargestFragmentChooser(preferOrganic=True)
_RDLogger.EnableLog('rdApp.*')

def keep_largest_fragment(mol):
    """
    Return the largest ORGANIC fragment (preferOrganic=True).
    "Organic" = contains at least one carbon atom.
    Among organic fragments, picks the one with the most heavy atoms.
    Examples:
        'CC(=O)[O-].[Na+]'        →  'CC(=O)[O-]'  (acetate, not Na+)
        'c1ccccc1.CCO'            →  'c1ccccc1'     (benzene, not ethanol)
        'CC(C)NCC(O)c1ccc(Cl)cc1.Cl'  →  free base  (not HCl)
    Returns the input unchanged if it is already a single fragment.
    """
    if mol is None:
        return None
    return _lfc.choose(mol)


# ============================================================
# TODO 1 — Parse SMILES into Mol objects
# ============================================================
# Apply smiles_to_mol() to df['SMILES'] and store in df['mol'].

df['mol'] = df['SMILES'].apply(___)

n_parsed = df['mol'].notna().sum()
print(f"Parsed:  {n_parsed} / {len(df)} molecules  ({len(df) - n_parsed} failed to parse)")

# ============================================================
# TODO 2 — Remove salts / select largest fragment
# ============================================================
# Apply keep_largest_fragment() to df['mol'] and overwrite df['mol'].

df['mol'] = df['mol'].apply(___)

# ============================================================
# PRE-WRITTEN: drop invalid rows and verify no multi-fragment molecules remain
# ============================================================
n_before  = len(df)
df_clean  = df[df['mol'].notna()].copy().reset_index(drop=True)
n_dropped = n_before - len(df_clean)

print(f"After standardisation: {len(df_clean)} molecules retained"
      + (f"  ({n_dropped} dropped)" if n_dropped else "  (none dropped — data was already clean)"))
print()

# fragment_count() uses RDKit's GetMolFrags() — imported from chem_utils.
# A '.' in SMILES is NOT a reliable fragment counter; use the Mol graph instead.
df_clean['n_fragments'] = df_clean['mol'].apply(fragment_count)
multi = df_clean[df_clean['n_fragments'] > 1]

print(f"Multi-fragment molecules after standardisation: {len(multi)}")
if len(multi) > 0:
    print("⚠️  Unexpected — keep_largest_fragment() should have resolved these:")
    display(multi[['COMPOUND_ID', 'SMILES', 'source', 'n_fragments']].head(5))
else:
    print("✅ All molecules are single fragments.")

# Clean up helper column
df_clean.drop(columns=['n_fragments'], inplace=True)


---
## Part C — Molecular Fingerprints (Morgan / ECFP4) ⏱ *~9 min*

### 🧬 What is a molecular fingerprint?

A **molecular fingerprint** is a fixed-length binary vector that encodes the structural features of a molecule as a series of 0s and 1s — a molecular "barcode".

**What are fingerprints used for?**

| Task | How fingerprints help |
|---|---|
| **Similarity search** | Compare two compounds via Tanimoto coefficient |
| **Virtual screening** | Rank a library against a known active |
| **Clustering / UMAP** | Project high-dimensional structure space to 2D |
| **Machine learning** | Use the bit vector as a feature matrix for QSAR models |

### 🔵 Morgan / ECFP4 — How they work

The Morgan algorithm generates **circular fingerprints** by expanding atom environments outward in concentric shells of bonds:

1. Assign each atom an initial identifier (element, charge, connectivity)
2. Iteratively hash each atom's identifier with its neighbours'
3. After `radius` iterations, collect all unique environment identifiers
4. Hash each to a bit position in a 2048-bit vector → set bit to 1

![Morgan fingerprint construction for aspirin (ECFP4)](../assets/Morgan_fingerprint_12293_2024_414_Fig3_HTML.png)

*Figure: Morgan (ECFP4) fingerprint construction. Source: Nguyen-Vo, T.-H. et al. Memetic Comp. 16, 519–536 (2024).*

**Parameters used:** `radius=2` (ECFP4), `nbits=2048`


In [ ]:
# ============================================================
# CELL D1 — Compute Morgan Fingerprints
# PRE-WRITTEN helper + TODO for the application
# ============================================================

# --- PRE-WRITTEN: helper function ---
def mol_to_fp(mol, radius=FP_RADIUS, nbits=FP_NBITS):
    """
    Compute a Morgan (ECFP-style) fingerprint as a numpy boolean array.
    Uses the new MorganGenerator API (RDKit >= 2022).
    Parameters
    ----------
    mol    : RDKit Mol object
    radius : int  — Morgan radius (2 = ECFP4, 3 = ECFP6)
    nbits  : int  — length of the bit vector (default: 2048)
    Returns
    -------
    numpy array of shape (nbits,), dtype uint8
    """
    from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
    gen = GetMorganGenerator(radius=radius, fpSize=nbits)
    fp  = gen.GetFingerprintAsNumPy(mol)
    return fp.astype(np.uint8)

# ============================================================
# TODO 2 — Apply mol_to_fp() to every molecule
# ============================================================
# Use a list comprehension over df_clean['mol'].
# Pass FP_RADIUS and FP_NBITS (already defined in Cell A1).
# Stack the result into a 2D numpy array called fp_matrix.
# Expected shape: (N, 2048)

fp_list   = [mol_to_fp(mol, radius=___, nbits=___) for mol in tqdm(df_clean['mol'], desc='Computing fingerprints')]
fp_matrix = np.vstack(___)

# --- PRE-WRITTEN: report ---
print(f"\nFingerprint matrix shape : {fp_matrix.shape}")
avg_bits_set = fp_matrix.sum(axis=1).mean()
print(f"Average bits set per molecule : {avg_bits_set:.1f} / {FP_NBITS}  ({100*avg_bits_set/FP_NBITS:.1f}% sparse)")
print("\n💡 Fingerprints are very sparse — this is why Tanimoto works well: shared bits are informative.")

In [ ]:
# ============================================================
# CELL D2 — Visualise the First 15 Fingerprints as a Heatmap
# PRE-WRITTEN: just run this cell
# ============================================================
# This makes the 'bit vector' concept tangible.

fig, ax = plt.subplots(figsize=(16, 3))
ax.imshow(fp_matrix[:15, :500], aspect='auto', cmap='Blues', interpolation='none')
ax.set_xlabel('Fingerprint bit index (first 500 of 2048)', fontsize=11)
ax.set_ylabel('Molecule index', fontsize=11)
ax.set_title('Morgan Fingerprint Bit Vectors — First 15 Molecules (first 500 bits)', fontsize=12)
ax.set_yticks(range(15))
ax.set_yticklabels([f"Mol {i} ({df_clean['source'].iloc[i]})" for i in range(15)], fontsize=9)
plt.tight_layout()
plt.show()
print("\n💡 Each dark pixel = bit set to 1 (structural motif present).")
print("   Most bits are 0 (white) — the fingerprint is sparse.")

### 💡 Why is the fingerprint so sparse?

The heatmap shows that most bits are **white (= 0)** — the fingerprint is **sparse**.

**Observed in our dataset:** ~47 bits set per molecule out of 2048 (≈ 2.3% density).

Three reasons stack on top of each other:

| Reason | Explanation |
|---|---|
| **Few environments per molecule** | ECFP4 assigns each atom up to 3 identifiers (radii 0, 1, 2). For ~30 heavy atoms → at most ~90 raw identifiers, already far fewer than 2048 |
| **Hash collisions compress further** | Multiple identifiers can hash to the same bit position, reducing ~90 raw identifiers down to ~47 set bits |
| **Binary encoding** | A substructure appearing twice still sets the bit only once |

**Why sparsity is a feature, not a bug:** Tanimoto similarity works precisely *because* fingerprints are sparse — shared bits between two sparse vectors are rare and therefore informative.


### 💬 Fingerprint Radius — Discussion

> *"What chemical information is encoded at `radius=2` vs. `radius=3`?  
> What is **lost** when we use a bit vector instead of a count vector?"*

| Parameter | What it captures | Risk |
|---|---|---|
| `radius=2` (ECFP4) | Rings + immediate substituents | Coarser, but robust |
| `radius=3` (ECFP6) | Larger scaffolds | More bit collisions → false similarities |
| **Bit** vector | Presence/absence of substructure | Loses frequency (how many times it appears) |
| **Count** vector | Frequency of substructure | More informative, less sparse, different similarity metrics |

---
## Part D — Physicochemical Descriptors (Lipinski) ⏱ *~5 min (part of Part C time)*

> 📌 **Physicochemical descriptors** encode ADME-relevant properties of a molecule.  
> Lipinski's Rule of 5 (Ro5) is the classic oral bioavailability filter.  
> MW ≤ 500 | LogP ≤ 5 | HBD ≤ 5 | HBA ≤ 10

In [ ]:
# ============================================================
# CELL C1 — Compute Lipinski Descriptors
# PRE-WRITTEN descriptor map + TODO for the application loop
# ============================================================

# --- PRE-WRITTEN: descriptor function map ---
DESCRIPTOR_FUNS = {
    'MW'       : Descriptors.ExactMolWt,
    'LogP'     : Descriptors.MolLogP,
    'HBD'      : rdMolDescriptors.CalcNumHBD,
    'HBA'      : rdMolDescriptors.CalcNumHBA,
    'TPSA'     : Descriptors.TPSA,
    'RotBonds' : rdMolDescriptors.CalcNumRotatableBonds,
    'RingCount': rdMolDescriptors.CalcNumRings,
}

# ============================================================
# TODO 3 — Compute descriptors for every molecule
# ============================================================
# For each descriptor name and its RDKit function,
# apply the function to the 'mol' column and store the result
# in df_clean as a new column named by the descriptor.

for desc_name, desc_fun in DESCRIPTOR_FUNS.items():
    df_clean[desc_name] = df_clean['mol'].apply(___)

# --- PRE-WRITTEN: summary stats ---
print("Descriptor summary statistics:")
display(df_clean[list(DESCRIPTOR_FUNS.keys())].describe().round(2))

In [ ]:
# ============================================================
# CELL C2 — Lipinski Descriptor Distributions
# PRE-WRITTEN: just run this cell
# ============================================================

desc_cols = list(DESCRIPTOR_FUNS.keys())
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(desc_cols):
    for source, colour in PALETTE.items():
        subset = df_clean[df_clean['source'] == source][col]
        axes[i].hist(subset, bins=30, alpha=0.6, color=colour, label=source, edgecolor='none')
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=8)

# Add Lipinski thresholds as vertical lines
lipinski_thresholds = {'MW': 500, 'LogP': 5, 'HBD': 5, 'HBA': 10}
for i, col in enumerate(desc_cols):
    if col in lipinski_thresholds:
        axes[i].axvline(lipinski_thresholds[col], color='red', linestyle='--', linewidth=1.5, label=f"Ro5: {lipinski_thresholds[col]}")
        axes[i].legend(fontsize=8)

axes[-1].axis('off')  # hide the 8th subplot
plt.suptitle('Lipinski Descriptor Distributions: MMV Malaria Box vs. AfroDb Natural Products',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL C3 — Lipinski Rule-of-5 Compliance Flag
# TODO 4: write the boolean Ro5 condition
# ============================================================

# Lipinski thresholds (constants)
MW_THRESH   = 500
LOGP_THRESH = 5
HBD_THRESH  = 5
HBA_THRESH  = 10

# ============================================================
# TODO 4 — Create a boolean column 'Ro5' that is True when
# the molecule passes ALL FOUR Lipinski rules.
# Fill in the comparison operators (<=):
# ============================================================

df_clean['Ro5'] = (
    (df_clean['MW']   ___ MW_THRESH)   &
    (df_clean['LogP'] ___ LOGP_THRESH) &
    (df_clean['HBD']  ___ HBD_THRESH)  &
    (df_clean['HBA']  ___ HBA_THRESH)
)

# --- PRE-WRITTEN: report ---
ro5_counts = df_clean.groupby('source')['Ro5'].value_counts(normalize=True).mul(100).round(1)
print("Lipinski Ro5 compliance rates by source:")
print(ro5_counts.rename('% of source').to_string())

# --- PRE-WRITTEN: scale descriptors for PCA (scaling does NOT affect fingerprint-based Tanimoto) ---
scaler    = StandardScaler()
desc_matrix_scaled = scaler.fit_transform(df_clean[desc_cols].fillna(df_clean[desc_cols].median()))
print(f"\n✅ Descriptor matrix scaled. Shape: {desc_matrix_scaled.shape}")
print("   Note: scaling is needed for PCA (distances matter) but NOT for Tanimoto (uses bit operations).")

### 💬 Stop & Discuss #1 — Data Hygiene

> **Question for the group:**  
> *"Suppose you are working with 200 SMILES from a local plant extract screen,  
> and 15 of them return `None` from `Chem.MolFromSmiles()`. What do you do?  
> Is it safe to silently drop them? What information might you lose?"*

**Think about:**
- Could the invalid SMILES be the most biologically active compounds in the set?
- Could the error be a notation issue (e.g., non-standard stereochemistry, salt notation) that can be fixed?
- What would you write in the Methods section of a paper about these dropped compounds?

---
## Part E — Pairwise Tanimoto Similarity ⏱ *~5 min*

> 📌 **Tanimoto coefficient** for two bit vectors A and B:
> 
> $$T(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{bits set in both}}{\text{bits set in either}}$$
> 
> - Range: 0 (nothing in common) → 1 (identical)
> - Rule of thumb: T > 0.85 = likely same scaffold; T < 0.4 = different chemical class
> - ⚠️ **Activity cliffs:** T > 0.9 molecules can have 1000× different activity!

In [ ]:
# ============================================================
# CELL E1 — Build Tanimoto Similarity Matrix (50-molecule subset)
# PRE-WRITTEN scaffold + TODO for the RDKit call
# ============================================================

# We use a 50-molecule subset for speed
N_SUBSET = 50

# numpy_to_rdkit_fp() converts numpy bit arrays to RDKit ExplicitBitVect,
# which is required by BulkTanimotoSimilarity(). Imported from chem_utils.
rdkit_fps_subset = [numpy_to_rdkit_fp(fp_list[i]) for i in range(N_SUBSET)]

# ── Pre-compute cache ──────────────────────────────────────────────────────
SIM_CACHE = '../data/cache_sim_matrix_50.npy'

# ============================================================
# TODO 5 — Compute the Tanimoto similarity matrix
# ============================================================
# For each fingerprint in rdkit_fps_subset,
# compute its similarity against ALL others using:
#    DataStructs.BulkTanimotoSimilarity(fp_query, fp_list)
# Store the result in `sim_matrix` (shape: N_SUBSET × N_SUBSET).

if os.path.exists(SIM_CACHE):
    sim_matrix = np.load(SIM_CACHE)
    print(f"✅ Loaded cached similarity matrix from {SIM_CACHE}")
else:
    sim_matrix = np.array([
        DataStructs.___(rdkit_fps_subset[i], rdkit_fps_subset)
        for i in range(N_SUBSET)
    ])
    np.save(SIM_CACHE, sim_matrix)
    print(f"✅ Similarity matrix computed and saved to {SIM_CACHE}")

print(f"Similarity matrix shape     : {sim_matrix.shape}")
print(f"Mean off-diagonal Tanimoto : {sim_matrix[np.triu_indices(N_SUBSET, k=1)].mean():.3f}")
print(f"Max off-diagonal Tanimoto  : {sim_matrix[np.triu_indices(N_SUBSET, k=1)].max():.3f}")


In [ ]:
# ============================================================
# CELL E2 — Tanimoto Heatmap with Source Annotation
# PRE-WRITTEN: just run this cell
# ============================================================

# Build source colour bar for rows/columns
sources_subset = df_clean['source'].iloc[:N_SUBSET].values
row_colours    = pd.Series(sources_subset).map(PALETTE)

fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
fig.colorbar(cax, ax=ax, label='Tanimoto Similarity', fraction=0.03)

# Annotate source boundaries
n_mmv   = (sources_subset == 'MMV').sum()
n_afro  = (sources_subset == 'AfroDb').sum()
if n_afro > 0:
    ax.axhline(n_mmv - 0.5, color='white', linewidth=2)
    ax.axvline(n_mmv - 0.5, color='white', linewidth=2)
    ax.text(n_mmv/2, -2, 'MMV', ha='center', color='#2196F3', fontweight='bold', fontsize=10)
    ax.text(n_mmv + n_afro/2, -2, 'AfroDb', ha='center', color='#4CAF50', fontweight='bold', fontsize=10)

ax.set_title(f'Pairwise Tanimoto Similarity — First {N_SUBSET} Compounds\n(ECFP4, radius=2, 2048 bits)', fontsize=12)
ax.set_xlabel('Compound index')
ax.set_ylabel('Compound index')
plt.tight_layout()
plt.show()

### 💬 Stop & Discuss #2 — Similarity Heatmap

> *"Look at the block structure in the heatmap.*  
> *1. Do you see a block of high similarity within the MMV compounds? What does that suggest about the library design?*  
> *2. How similar are the AfroDb natural products to the MMV compounds overall?*  
> *3. Can you find a pair of molecules with high Tanimoto but potentially very different activity? This is an **activity cliff**.*"

---

**⚡ EXTENSION E1** *(skip if short on time)*
```python
# Compute the FULL similarity matrix (all compounds) and find the most similar non-identical pair.
# Plot the distribution of all off-diagonal Tanimoto values.
# What does the shape of this distribution tell you about library diversity?
# (A flat distribution = diverse; a peak near 1 = redundant library)
```

---
## Part F — Dimensionality Reduction: PCA ⏱ *~4 min*

> 📌 **Principal Component Analysis (PCA)** finds the directions of maximum variance in the data.  
> It is linear, fast, and deterministic — but it cannot capture non-linear structure.  
> We run PCA on the **scaled descriptor matrix** (Lipinski descriptors), not the fingerprint matrix.

In [ ]:
# ============================================================
# CELL F1 — PCA: Scree Plot (explained variance)
# PRE-WRITTEN: just run this cell
# ============================================================

pca_full = PCA(n_components=min(len(desc_cols), 7), random_state=RANDOM_SEED)
pca_full.fit(desc_matrix_scaled)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
       pca_full.explained_variance_ratio_ * 100, color='steelblue', edgecolor='white')
ax.plot(range(1, len(pca_full.explained_variance_ratio_) + 1),
        np.cumsum(pca_full.explained_variance_ratio_) * 100, 'ro-', label='Cumulative')
ax.axhline(80, linestyle='--', color='grey', alpha=0.7, label='80% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)')
ax.set_title('PCA Scree Plot — Lipinski Physicochemical Descriptors')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL F2 — PCA: Fit 2D projection
# TODO 6: fit the PCA model and project the data
# ============================================================

# ============================================================
# TODO 6 — Fit a PCA model with 2 components on desc_matrix_scaled
# and transform the data. Store the result in pca_coords.
# Use random_state=RANDOM_SEED.
# ============================================================

pca = PCA(n_components=___, random_state=___)
pca_coords = pca.___(desc_matrix_scaled)

# Store in DataFrame
df_clean['PCA_1'] = pca_coords[:, 0]
df_clean['PCA_2'] = pca_coords[:, 1]

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
print(f"PC1 explained variance: {var1:.1f}%")
print(f"PC2 explained variance: {var2:.1f}%")
print(f"Total (2 PCs)          : {var1+var2:.1f}%")
print()
print("💡 If this is low (e.g., <50%), it means 2D PCA misses most of the chemical variation.")
print("   This is one reason why UMAP (non-linear) often gives more informative maps.")

# --- PRE-WRITTEN: scatter plot ---
fig, ax = plt.subplots(figsize=(8, 6))
for source, colour in PALETTE.items():
    mask = df_clean['source'] == source
    ax.scatter(df_clean.loc[mask, 'PCA_1'], df_clean.loc[mask, 'PCA_2'],
               c=colour, label=source, s=30, alpha=0.7, edgecolors='none')
ax.set_xlabel(f'PC1 ({var1:.1f}% variance)')
ax.set_ylabel(f'PC2 ({var2:.1f}% variance)')
ax.set_title('PCA of Lipinski Descriptors\nMMV Malaria Box vs. AfroDb Natural Products')
ax.legend(title='Dataset')
plt.tight_layout()
plt.show()

### 📊 PCA — What to look for

> 💡 **Scree plot:** The elbow (where bars flatten) shows how many PCs are informative. For 6–7 Lipinski descriptors, PC1 + PC2 typically capture ~55–65% of variance — enough for a useful 2D view.

**PC1 ≈ molecular size** (MW, HBA, RotBonds co-vary) · **PC2 ≈ hydrophobicity vs. polarity** (LogP vs. HBD/TPSA)

| What you see | What it means |
|---|---|
| MMV + AfroDb overlap in the centre | Both are broadly drug-like (similar MW/LogP ranges) |
| AfroDb extends further along PC1 | Larger, more complex NPs (terpenoids, alkaloids) pushing the Ro5 MW limit |
| AfroDb polar stripe at high PC2 | Flavonoids and glycosides — hydrophilic NP classes |

> ⚠️ **Key caveat:** PCA here answers *“are they physically similar?”* — **not** *“are they structurally similar?”*. Two molecules can share identical MW, LogP, and HBD yet have Tanimoto < 0.2. That is why Part G uses UMAP on fingerprints.


### 💬 Stop & Discuss #2b — PCA Limitations

> *“The PCA plot shows MMV and AfroDb overlapping quite heavily.*  
> *1. Does this mean the two libraries are **structurally** similar? Why or why not?*  
> *2. PC1 captures mostly molecular size. What would this PCA look like for a fragment-only dataset (MW < 250)?*  
> *3. Why would PCA on the 2048-bit Morgan fingerprint matrix be a poor choice — and why is UMAP better?”*


---
## Part G — Dimensionality Reduction: UMAP ⏱ *~8 min*

> 📌 **UMAP (Uniform Manifold Approximation and Projection)** is a non-linear dimensionality reduction  
> method. It builds a graph of nearest neighbors in high-dimensional space and optimizes a 2D layout  
> that preserves local neighborhoods.
>
> ⚠️ **Critical warnings:**
> 1. **Distances between clusters are NOT interpretable** — only within-cluster structure is reliable.
> 2. **Always set `random_state`** — UMAP uses randomness; different seeds give different layouts.
> 3. `metric='jaccard'` is appropriate for binary fingerprints (Jaccard distance = 1 − Tanimoto).

| Hyperparameter | What it controls | Effect of increasing |
|---|---|---|
| `n_neighbors` | Size of local neighborhood | Smoother, more global structure |
| `min_dist` | How tightly points are packed | Higher = more spread out |
| `metric` | Distance function | Must match data type |

In [ ]:
# ============================================================
# CELL G1 — UMAP on Morgan Fingerprints
# TODO 7: instantiate and fit the UMAP reducer
# ============================================================

# ============================================================
# TODO 7 — Instantiate umap.UMAP() with the following parameters:
#   n_neighbors = 15
#   min_dist    = 0.1
#   metric      = 'jaccard'   ← why jaccard? because for binary vectors,
#                               jaccard distance = 1 - Tanimoto similarity!
#   n_components = 2
#   random_state = RANDOM_SEED
# Then call fit_transform() on fp_matrix.
# ============================================================

reducer = umap.UMAP(
    n_neighbors  = ___,
    min_dist     = ___,
    metric       = ___,
    n_components = ___,
    random_state = ___
)

# ── Pre-compute cache ─────────────────────────────────────────────────────────
# cache_umap_coords.npy — what it contains and how it was generated
# ─────────────────────────────────────────────────────────────────────────────
#
#  Input data  : data/malaria_box_afrodb_combined.csv  (1 303 compounds)
#                  400 × MMV Malaria Box  (confirmed P. falciparum actives)
#                  903 × AfroDb African natural products
#
#  Preprocessing applied before UMAP:
#    1. Chem.MolFromSmiles()                       — parse SMILES → RDKit Mol
#    2. LargestFragmentChooser(preferOrganic=True) — salt removal
#    3. Drop None rows                             — result: 1 303 molecules
#
#  Fingerprints:
#    GetMorganGenerator(radius=2, fpSize=2048)     — ECFP4, 2048-bit, dtype uint8
#    → fp_matrix  shape (1303, 2048)
#
#  UMAP parameters (must match Cell A1 constants):
#    n_neighbors=15, min_dist=0.1, metric='jaccard', random_state=42
#
#  Output:
#    umap_coords  shape (1303, 2)  float32
#    Row i  ↔  df_clean.iloc[i]  (same order as the combined CSV)
#
#  To regenerate after changing RANDOM_SEED, FP_RADIUS, or the dataset:
#    rm data/cache_umap_coords.npy
#    conda run -n cheminf python scripts/generate_caches.py
# ─────────────────────────────────────────────────────────────────────────────
UMAP_CACHE = '../data/cache_umap_coords.npy'

if os.path.exists(UMAP_CACHE):
    umap_coords = np.load(UMAP_CACHE)
    print(f"✅ Loaded cached UMAP coordinates from {UMAP_CACHE}")
    print(f"   Shape: {umap_coords.shape}  (delete file to force recomputation)")
    print(f"   Data:  1303 compounds = 400 MMV + 903 AfroDb")
    print(f"   Model: ECFP4 (r=2, 2048 bits) → UMAP(jaccard, n_neighbors=15, seed=42)")
    # Fit the reducer so reducer.transform() works in Extension 4.
    # We skip saving umap_coords (already cached), but the model must be fitted
    # for transform() to have access to the learned nearest-neighbour graph.
    print("   Fitting reducer on fp_matrix (required for Extension 4 transform)...")
    reducer.fit(fp_matrix)
    print("   ✅ reducer fitted.")
else:
    print("Running UMAP... (may take ~30 seconds on a laptop, up to 2 min on slow hardware)")
    umap_coords = reducer.fit_transform(fp_matrix)
    np.save(UMAP_CACHE, umap_coords)
    print(f"✅ UMAP complete. Embedding saved to {UMAP_CACHE}")

df_clean['UMAP_1'] = umap_coords[:, 0]
df_clean['UMAP_2'] = umap_coords[:, 1]
print(f"   Embedding shape: {umap_coords.shape}")


In [ ]:
# ============================================================
# CELL G2 — Side-by-Side: PCA vs UMAP
# PRE-WRITTEN: just run this cell
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (x_col, y_col, title) in zip(axes, [
    ('PCA_1',  'PCA_2',  f'PCA (Lipinski descriptors)\n{var1:.1f}% + {var2:.1f}% = {var1+var2:.1f}% variance'),
    ('UMAP_1', 'UMAP_2', 'UMAP (ECFP4 fingerprints, jaccard)\n n_neighbors=15, min_dist=0.1')
]):
    for source, colour in PALETTE.items():
        mask = df_clean['source'] == source
        ax.scatter(df_clean.loc[mask, x_col], df_clean.loc[mask, y_col],
                   c=colour, label=source, s=25, alpha=0.75, edgecolors='none')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(x_col.replace('_', ' '))
    ax.set_ylabel(y_col.replace('_', ' '))
    ax.legend(title='Dataset', fontsize=9)

plt.suptitle('Chemical Space: PCA vs. UMAP\nMMV Malaria Box (blue) vs. AfroDb Natural Products (green)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print()
print("⚠️  Reminder: In UMAP, distances BETWEEN clusters are NOT meaningful.")
print("   Only structure WITHIN clusters (tight groups = similar molecules) is interpretable.")

### 🗺️ Cell G3 — Interactive 3D UMAP *(pre-written — run & explore)*

The 2D projection flattens one dimension of variation. A **3-component UMAP** preserves more of the manifold structure and lets you rotate the map to find clusters that are hidden in the flat view.

> 🖱️ **Tip:** Use your mouse to rotate the 3-D plot. Look for clusters that are well separated in 3D but overlapping in the 2D view — those are chemically distinct subsets that 2D projection "squashes" together.


In [ ]:
# ============================================================
# CELL G3 — Interactive 3D UMAP (Plotly)
# ============================================================
import plotly.express as px
import umap as umap_lib

reducer_3d = umap_lib.UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,
    metric="jaccard",
    random_state=RANDOM_SEED,
)
umap_3d = reducer_3d.fit_transform(fp_matrix)

df_clean["UMAP3_1"] = umap_3d[:, 0]
df_clean["UMAP3_2"] = umap_3d[:, 1]
df_clean["UMAP3_3"] = umap_3d[:, 2]

fig = px.scatter_3d(
    df_clean,
    x="UMAP3_1", y="UMAP3_2", z="UMAP3_3",
    color="source",
    color_discrete_map=PALETTE,
    hover_data={"COMPOUND_ID": True, "LogP": ":.2f", "MW": ":.0f",
                "UMAP3_1": False, "UMAP3_2": False, "UMAP3_3": False},
    symbol="source",
    opacity=0.75,
    title="3D UMAP of Morgan Fingerprints — coloured by dataset",
    labels={"UMAP3_1": "UMAP 1", "UMAP3_2": "UMAP 2", "UMAP3_3": "UMAP 3"},
    width=900, height=700,
)
fig.update_traces(marker=dict(size=3))
fig.show()


### 💬 Stop \& Discuss #3 — PCA vs. UMAP

> *"Compare the PCA and UMAP plots:*
> 1. Which projection shows clearer cluster separation between AfroDb and MMV compounds?
> 2. What does it mean when two points are close together in UMAP space?
> 3. **3D view (Cell G3):** Rotate the 3D map. Do you see clusters that were hidden in the 2D projection?"*


---
## Part H — Property-Based Visualization ⏱ *~5 min*

> 📌 A scatter plot is just a canvas. Scientific insight comes from colouring by  
> biologically meaningful properties. We will create three panels:
> 1. Coloured by **pIC50** (anti-malarial potency)
> 2. Coloured by **LogP** (hydrophobicity)
> 3. Coloured by **Lipinski Ro5 compliance** (categorical: pass/fail)

In [ ]:
# ============================================================
# CELL H1 — UMAP Coloured by pIC50
# TODO 8: fill in the scatter plot call
# ============================================================

# Only MMV compounds have pIC50 values
mmv_mask = df_clean['source'] == 'MMV'
df_mmv   = df_clean[mmv_mask].copy()

fig, ax = plt.subplots(figsize=(9, 7))

# Background: all compounds in grey
ax.scatter(df_clean['UMAP_1'], df_clean['UMAP_2'], c='lightgrey', s=15, alpha=0.4, edgecolors='none', zorder=1)

# ============================================================
# TODO 8 — Overlay MMV compounds coloured by pIC50.
# Use:
#   x = df_mmv['UMAP_1'],  y = df_mmv['UMAP_2']
#   c = df_mmv['pIC50']    ← this column holds the activity values
#   cmap = 'plasma'        ← perceptually uniform colormap
#   s=40, alpha=0.85, edgecolors='none', zorder=2
# Store the scatter return value as `sc` to attach the colorbar.
# ============================================================

sc = ax.scatter(
    df_mmv[___],      # x axis
    df_mmv[___],      # y axis
    c    = ___,       # colour by pIC50
    cmap = ___,
    s=40, alpha=0.85, edgecolors='none', zorder=2
)

# --- PRE-WRITTEN: colorbar and labels ---
cbar = fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('pIC50 (anti-malarial, P. falciparum)', fontsize=10)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('UMAP Chemical Space — Coloured by Anti-Malarial Potency (pIC50)\nMMV Malaria Box | ECFP4 | jaccard distance', fontsize=11)
ax.text(0.02, 0.97, 'Grey = AfroDb natural products\n(no pIC50 available)',
        transform=ax.transAxes, fontsize=9, va='top', color='grey')
plt.tight_layout()
plt.show()

### 💬 Stop & Discuss #4 — Activity Cliffs

> *"In your pIC50-coloured UMAP:*  
> *1. Can you find a region where neighbouring points have very different colours (yellow next to purple)?*  
>    *This is an **activity cliff** — structurally similar molecules with dramatically different potency.*  
> *2. What does this mean for a medicinal chemist trying to optimize a lead compound?*  
> *3. Would a 'cluster = similar activity' assumption be dangerous here?"*

In [ ]:
# ============================================================
# CELL H2 — UMAP Coloured by LogP and Ro5 Compliance
# PRE-WRITTEN: just run this cell — focus on interpretation
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: LogP continuous colormap
sc1 = axes[0].scatter(
    df_clean['UMAP_1'], df_clean['UMAP_2'],
    c=df_clean['LogP'], cmap='RdYlGn_r',
    s=25, alpha=0.8, edgecolors='none'
)
cbar1 = fig.colorbar(sc1, ax=axes[0], fraction=0.03)
cbar1.set_label('LogP (hydrophobicity)')
axes[0].set_title('UMAP Coloured by LogP\n(green = hydrophilic, red = lipophilic)')
axes[0].set_xlabel('UMAP 1')
axes[0].set_ylabel('UMAP 2')

# Panel 2: Lipinski Ro5 compliance (categorical)
ro5_colours = df_clean['Ro5'].map({True: '#2196F3', False: '#F44336'})  # blue=pass, red=fail
axes[1].scatter(
    df_clean['UMAP_1'], df_clean['UMAP_2'],
    c=ro5_colours, s=25, alpha=0.8, edgecolors='none'
)
pass_patch = mpatches.Patch(color='#2196F3', label='Ro5 Pass')
fail_patch = mpatches.Patch(color='#F44336', label='Ro5 Fail')
axes[1].legend(handles=[pass_patch, fail_patch], title='Lipinski Ro5')
axes[1].set_title('UMAP Coloured by Lipinski Ro5 Compliance\n(blue = passes all 4 rules, red = fails ≥1)')
axes[1].set_xlabel('UMAP 1')
axes[1].set_ylabel('UMAP 2')

plt.suptitle('Property Islands in Chemical Space', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print()
print("💡 'Property islands': regions of chemical space with consistently high or low LogP")
print("   often reflect scaffold-level trends — e.g., aromatic-rich scaffolds cluster in high-LogP regions.")
print()
print("💡 Ro5 failures in AfroDb: natural products often violate Lipinski rules because they evolved")
print("   for activity, not for oral bioavailability. This is the 'beyond Ro5' research space.")

### ✏️ Student Observation Cell

**Double-click this cell to edit it.** Write 2–3 sentences about what you observe in the plots above:

1. **LogP map:** *[Your observation here — do you see property islands? Where are the most hydrophobic compounds?]*

2. **Ro5 compliance map:** *[Are Ro5 failures concentrated in a specific region? Which source (MMV vs AfroDb) fails more?]*

3. **Connection to drug discovery:** *[What would you prioritize if you were looking for an orally bioavailable anti-malarial lead?]*

---
## Part I — Capstone: AfroDb vs. MMV Chemical Space Overlay ⏱ *~5 min*

> 📌 **The key question:** Which African natural products (AfroDb) fall inside or near the  
> MMV Malaria Box chemical space? These are priority candidates for anti-malarial screening.
>
> This figure is designed to be **publication-quality** — suitable for a thesis or poster.

> ⚠️ **Methodological note — joint UMAP (this cell) vs. projected UMAP (Extension 4):**  
> Here, MMV and AfroDb fingerprints are pooled into a **single matrix** before calling `fit_transform()`.  
> Both libraries jointly shape the UMAP coordinate system.  
> This is ideal for an **exploratory side-by-side comparison**.
>
> Extension 4 takes a more rigorous approach: the UMAP is fitted **on MMV only**, and AfroDb is projected into that fixed coordinate frame using `reducer.transform()`. This answers: *"Where do African natural products land within the known anti-malarial chemical space?"* — closer to how virtual screening is done in practice.
>
> 👉 See **Extension 4** for the projected version.

In [ ]:
# ============================================================
# CELL I1 — Capstone: Publication-Quality Chemical Space Map
# PRE-WRITTEN figure + TODO 9: add legend and labels
# ============================================================

# Separate the two datasets
df_mmv_all  = df_clean[df_clean['source'] == 'MMV']
df_afro_all = df_clean[df_clean['source'] == 'AfroDb']

fig, ax = plt.subplots(figsize=(11, 8), dpi=150)

# MMV Malaria Box: circles, sized by MW, coloured by LogP
sc_mmv = ax.scatter(
    df_mmv_all['UMAP_1'], df_mmv_all['UMAP_2'],
    c     = df_mmv_all['LogP'],
    cmap  = 'coolwarm',
    s     = (df_mmv_all['MW'] / 500 * 80).clip(10, 120),  # size scaled by MW
    alpha = 0.75,
    marker    = 'o',
    edgecolors= 'white',
    linewidths= 0.3,
    label = 'MMV Malaria Box (circles)',
    zorder= 2
)

# AfroDb Natural Products: triangles, sized by MW, coloured by LogP (same scale)
sc_afro = ax.scatter(
    df_afro_all['UMAP_1'], df_afro_all['UMAP_2'],
    c     = df_afro_all['LogP'],
    cmap  = 'coolwarm',
    vmin  = df_mmv_all['LogP'].min(),
    vmax  = df_mmv_all['LogP'].max(),
    s     = (df_afro_all['MW'] / 500 * 80).clip(10, 120),
    alpha = 0.85,
    marker    = '^',
    edgecolors= 'black',
    linewidths= 0.5,
    label = 'AfroDb Natural Products (triangles)',
    zorder= 3
)

cbar = fig.colorbar(sc_mmv, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('LogP (hydrophobicity)', fontsize=10)

# ============================================================
# TODO 9 — Add a legend, axis labels, and a descriptive title.
# The legend should distinguish MMV circles from AfroDb triangles.
# ============================================================

ax.legend(title=___, fontsize=10, title_fontsize=10, loc='lower right')
ax.set_xlabel(___, fontsize=12)
ax.set_ylabel(___, fontsize=12)
ax.set_title(___, fontsize=13)

# --- PRE-WRITTEN: size legend annotation ---
for mw_val, label in [(300, 'MW=300'), (500, 'MW=500'), (700, 'MW=700')]:
    ax.scatter([], [], s=mw_val/500*80, c='grey', alpha=0.6, label=label)
ax.legend(fontsize=9, loc='lower left')

plt.tight_layout()
plt.savefig('../data/chemical_space_map.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figure saved to: data/chemical_space_map.png")

### ✏️ Reflection — The African Drug Discovery Opportunity

**Double-click this cell to edit it.** Answer the following in 3–5 sentences:

> *"Looking at the final chemical space map: which AfroDb natural products (triangles) fall **inside**  
> or **near** the MMV Malaria Box cluster? What might this suggest for lead identification against  
> Plasmodium falciparum? Are any AfroDb compounds in a completely unexplored region — and what  
> could that mean?"*

**Your interpretation:**  
*[Write 3–5 sentences here]*

---

**Key takeaways from this workshop:**

| Concept | What we learned |
|---|---|
| SMILES sanitization | Always validate — silent failures contaminate all downstream results |
| Morgan fingerprints | Encode local chemical topology as sparse bit vectors; radius matters |
| Tanimoto similarity | Standard chemical similarity metric; beware activity cliffs |
| PCA | Linear, fast, interpretable axes; limited for non-linear chemical structure |
| UMAP | Non-linear, reveals clusters; distances between clusters are NOT interpretable |
| Property colouring | Reveals property islands; connects structural diversity to ADME/activity |
| Cultural relevance | AfroDb natural products occupy distinctive chemical space — a research opportunity |

In [ ]:
# ============================================================
# EXTENSION 1 — Murcko Scaffold Extraction
# TODO EXT1: extract scaffold and count unique scaffolds
# ============================================================

from rdkit.Chem.Scaffolds import MurckoScaffold

def get_murcko_smiles(mol):
    """Return the Murcko scaffold SMILES, or 'no_scaffold' for acyclic molecules."""
    try:
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        smi = Chem.MolToSmiles(scaffold)
        return smi if smi else 'no_scaffold'
    except:
        return 'no_scaffold'

# TODO EXT1: Apply get_murcko_smiles to df_clean['mol']
# Store in df_clean['scaffold']
df_clean['scaffold'] = df_clean['mol'].apply(___)

n_unique_scaffolds = df_clean['scaffold'].nunique()
print(f"Unique Murcko scaffolds: {n_unique_scaffolds} across {len(df_clean)} molecules")
print(f"Scaffold diversity ratio: {n_unique_scaffolds / len(df_clean):.2f}")
print("  (1.0 = maximally diverse; <0.5 = scaffold-biased library)")

# PRE-WRITTEN: Colour UMAP by top-10 scaffolds
top_scaffolds = df_clean['scaffold'].value_counts().head(10).index
df_clean['scaffold_label'] = df_clean['scaffold'].apply(
    lambda s: s[:20] + '...' if s in top_scaffolds else 'other'
)

fig, ax = plt.subplots(figsize=(10, 8))
palette_ext = sns.color_palette('tab10', n_colors=11)
for i, label in enumerate(list(top_scaffolds[:10]) + ['other']):
    mask = df_clean['scaffold_label'].str.startswith(label[:20])
    ax.scatter(df_clean.loc[mask, 'UMAP_1'], df_clean.loc[mask, 'UMAP_2'],
               c=[palette_ext[i]], s=20, alpha=0.7, label=f'SC{i+1}' if label != 'other' else 'other',
               edgecolors='none')
ax.legend(title='Scaffold', fontsize=8, ncol=2)
ax.set_title('UMAP Coloured by Murcko Scaffold\nDo scaffold clusters match UMAP clusters?')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
plt.tight_layout()
plt.show()

---
## ⚡ EXTENSION 2 — Interactive Molecule Browser with mols2grid

*Requires: `pip install mols2grid`*

> **Goal:** Build an interactive, searchable grid of all 1 303 compounds where each cell shows the **2D structure**, source, MW, LogP, and Ro5 flag.  
> `mols2grid` renders structures efficiently through virtual scrolling — no memory bloat regardless of dataset size.  
> This is the standard tool for rapid visual triage of screening results in industry.


In [ ]:
# ============================================================
# EXTENSION 2 — Interactive Molecule Browser with mols2grid
# TODO EXT2: fill in the subset argument
# ============================================================

try:
    import mols2grid
except ImportError:
    print("Install mols2grid: pip install mols2grid")
    raise

# Prepare display dataframe
# mols2grid expects a 'SMILES' column (already present) and optional extra columns
display_cols = ['SMILES', 'COMPOUND_ID', 'source', 'MW', 'LogP', 'Ro5']
df_grid = df_clean[display_cols].copy()
df_grid['pIC50'] = df_clean.get('pIC50', None)

# Round numeric columns for cleaner card display
df_grid['MW']    = df_grid['MW'].round(1)
df_grid['LogP']  = df_grid['LogP'].round(2)
df_grid['pIC50'] = df_grid['pIC50'].round(2)

# Convert boolean Ro5 flag to readable label
df_grid['Ro5'] = df_grid['Ro5'].map({True: 'Ro5 Pass', False: 'Ro5 Fail'})

# ============================================================
# TODO EXT2 — Launch the interactive grid.
# Fill in the subset argument with the columns you want on each card.
# Hint: subset=['COMPOUND_ID', 'source', 'MW', 'LogP', 'Ro5', 'pIC50']
# ============================================================

from IPython.display import HTML

result = mols2grid.display(
    df_grid,
    smiles_col = 'SMILES',
    subset     = ___,  # TODO EXT2: fill in the column list
    tooltip    = ['COMPOUND_ID', 'SMILES', 'source', 'MW', 'LogP', 'Ro5', 'pIC50'],
    template   = 'interactive',
    size       = (160, 120),
    n_cols     = 5,
    output     = 'html',
)
HTML(result.data)


---
## ⚡ EXTENSION 3 — Activity Cliffs: Structurally Similar, Biologically Opposite

*~10 min · Requires: Parts A–D complete (for imports and `mol_to_fp`)*

> 📌 An **activity cliff** is a pair of molecules that are structurally very similar
> (Tanimoto ≥ 0.9) but have **opposite** biological outcomes — one Active, one Inactive.
>
> We use the **PubChem AID 2302** dataset (P. *falciparum* Dd2, whole-cell assay, 2 µM cut-off):
> a 2 000-compound screen with binary Active/Inactive labels — much better suited for cliff
> detection than the MMV Malaria Box (which contains only confirmed actives, so all pIC50 values
> are narrow and high).
>
> **What you will do:**
> 1. Load AID 2302, parse SMILES, compute ECFP4 fingerprints
> 2. Build RDKit `ExplicitBitVect` objects and compute all pairwise Tanimoto similarities
> 3. Find pairs where Tanimoto ≥ 0.9 **and** activities differ (Active ↔ Inactive)
> 4. Draw the top-5 cliff pairs side-by-side

> 💡 **Why AID 2302 and not the MMV Malaria Box?**
> The Malaria Box was curated from confirmed actives — all 400 compounds already passed an
> EC₅₀ < 4 µM screen. The narrow pIC50 range (std ≈ 0.44) makes large potency jumps between
> similar structures essentially impossible. AID 2302 covers the full Active/Inactive spectrum
> and is ideal for observing true activity cliffs.

> 💡 **Why a threshold of Tanimoto ≥ 0.9?**
> The 0.9 cut-off is the widely-used convention for defining "very high structural similarity"
> in cheminformatics (Martin *et al.*, *J. Med. Chem.* 2002). At T ≥ 0.9, compounds typically
> share the same core scaffold and differ by only one or two substituents — small enough that
> any large activity difference is chemically surprising and scientifically meaningful.
>
> Lower thresholds (e.g. T ≥ 0.7) would include structurally more distant pairs, where an
> activity difference is less surprising and harder to interpret. Higher thresholds (e.g. T ≥ 0.95)
> would capture near-identical compounds, but at the cost of finding very few or no pairs in a
> 2 000-compound dataset.

In [ ]:
# ============================================================
# EXTENSION 3 — Activity Cliffs (PubChem AID 2302)
# TODO EXT3: find cliff pairs and draw them
# ============================================================

from rdkit import RDLogger
from rdkit.DataStructs import ExplicitBitVect
from rdkit.Chem import Draw
from IPython.display import display as ipy_display

RDLogger.DisableLog('rdApp.*')

# --- PRE-WRITTEN: Load AID 2302, parse SMILES, and standardise (remove salts) ---
AID_PATH = '../data/pubchem_aid2302_2k.csv'
df_aid = pd.read_csv(AID_PATH)
df_aid['mol'] = df_aid['SMILES'].apply(smiles_to_mol)
df_aid['mol'] = df_aid['mol'].apply(keep_largest_fragment)  # remove salts / keep largest fragment
df_aid = df_aid[df_aid['mol'].notna()].copy().reset_index(drop=True)
print(f"AID 2302: {len(df_aid)} compounds parsed and standardised")
print(f"  Active: {(df_aid['activity']=='Active').sum()}  |  Inactive: {(df_aid['activity']=='Inactive').sum()}")

# Compute fingerprints using the same mol_to_fp() from Part D
aid_fp_matrix = np.vstack([
    mol_to_fp(m, FP_RADIUS, FP_NBITS)
    for m in tqdm(df_aid['mol'], desc='AID fingerprints')
])

# Build RDKit ExplicitBitVect objects for BulkTanimotoSimilarity
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit import DataStructs as _DS
_gen = GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_NBITS)
fps_aid = [_gen.GetFingerprint(m) for m in df_aid['mol']]

# ============================================================
# TODO EXT3-A — Compute all-vs-all Tanimoto similarity matrix.
# Use DataStructs.BulkTanimotoSimilarity in a loop.
# Store as 2D numpy array `sim_matrix` (shape: N x N).
# ============================================================
n = len(fps_aid)
sim_matrix = np.zeros((n, n))
for i in range(n):
    sim_matrix[i] = DataStructs.___(fps_aid[i], fps_aid)   # TODO EXT3-A

# ============================================================
# TODO EXT3-B — Find activity cliff pairs using numpy (fast).
# A cliff pair (i, j) satisfies ALL of:
#   i < j                                      (upper triangle only)
#   sim_matrix[i, j] >= 0.9                    (structurally very similar)
#   activity[i] != activity[j]                 (opposite outcomes)
#
# HINT: use np.triu to extract the upper triangle, then np.where
# to find pairs above the threshold, then filter by activity.
# ============================================================
activity_arr = df_aid['activity'].values

# Step 1: upper-triangle mask (i < j only) above threshold
triu = np.triu(sim_matrix, k=1)                       # zeros on/below diagonal
rows, cols = np.where(triu >= ___)                    # TODO EXT3-B: threshold

# Step 2: keep only pairs with opposite activity labels
opp = activity_arr[rows] != activity_arr[cols]        # TODO EXT3-B: use != to find opposite activities
rows, cols = rows[opp], cols[opp]

cliff_pairs = sorted(
    [{'i': int(i), 'j': int(j), 'tanimoto': sim_matrix[i, j]} for i, j in zip(rows, cols)],
    key=lambda x: x['tanimoto'], reverse=True
)[:5]

cliff_pairs = sorted(cliff_pairs, key=lambda x: x['tanimoto'], reverse=True)[:5]
print(f"\nActivity cliff pairs found (Tan>=0.9, Active<->Inactive): {len(cliff_pairs)}")
for cp in cliff_pairs:
    act_i = df_aid.iloc[cp['i']]['activity']
    act_j = df_aid.iloc[cp['j']]['activity']
    cid_i = df_aid.iloc[cp['i']]['CID']
    cid_j = df_aid.iloc[cp['j']]['CID']
    print(f"  CID {cid_i} ({act_i}) vs CID {cid_j} ({act_j})  Tan={cp['tanimoto']:.3f}")

# --- PRE-WRITTEN: Draw the top-5 cliff pairs ---
if cliff_pairs:
    for k, cp in enumerate(cliff_pairs[:5]):
        mol_a  = df_aid.iloc[cp['i']]['mol']
        mol_b  = df_aid.iloc[cp['j']]['mol']
        act_a  = df_aid.iloc[cp['i']]['activity']
        act_b  = df_aid.iloc[cp['j']]['activity']
        cid_a  = df_aid.iloc[cp['i']]['CID']
        cid_b  = df_aid.iloc[cp['j']]['CID']
        img = Draw.MolsToGridImage(
            [mol_a, mol_b],
            molsPerRow=2,
            subImgSize=(350, 280),
            legends=[
                f"CID {cid_a}\n{act_a}",
                f"CID {cid_b}\n{act_b}\n(Tanimoto = {cp['tanimoto']:.3f})",
            ],
        )
        print(f"\n--- Cliff pair {k+1} (Tan={cp['tanimoto']:.3f}) ---")
        ipy_display(img)
else:
    print("No cliff pairs found at Tan>=0.9 — try lowering to 0.85")

RDLogger.EnableLog('rdApp.*')

---
## ⚡ EXTENSION 4 — AfroDb Overlay: African Natural Products in the Malaria Box Chemical Space

*~10 min · Requires: `reducer` (fitted UMAP from Part G), `fp_matrix` and `df_clean` from Parts D & G*

> 📌 **How this differs from Part I:**
>
> | | Part I | Extension 4 (this cell) |
> |---|---|---|
> | **UMAP fitted on** | MMV + AfroDb combined | MMV only |
> | **AfroDb role** | Co-shapes the map | Read-only overlay |
> | **Key call** | `fit_transform(mmv + afrodb)` | `transform(afrodb)` |
> | **Question answered** | "How do the two libraries compare?" | "Where do NPs land *in drug space*?" |
> | **Analogy** | Redrawing the map with both continents | Placing pins on an existing map |
>
> The projected approach is **more rigorous for virtual screening**: the coordinate frame is anchored to the known anti-malarial library, so AfroDb compounds near an MMV cluster are making a structurally meaningful claim — not just an artefact of joint embedding.

> 💡 **What to expect:** AfroDb compounds will only *partially* overlap with the MMV clusters.
> This is **correct and expected** — natural products have very different scaffolds from synthetic
> drug-like compounds. The triangles that fall *outside* the grey MMV region represent
> **unexplored chemical space**: structurally novel compounds that no anti-malarial drug screen
> has reached yet.

> **Fill in one key line (TODO EXT4).**  
> The rest — loading data, computing fingerprints, plotting — is pre-written.

In [ ]:
# ============================================================
# EXTENSION 4 — AfroDb Overlay on Malaria Box UMAP
# PRE-WRITTEN scaffold + TODO EXT4 for the key projection line
# ============================================================

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# --- PRE-WRITTEN: Load AfroDb, parse SMILES, compute fingerprints ---
afrodb_path = '../data/afrodb_subset.csv'
df_afrodb   = pd.read_csv(afrodb_path)

# Apply the same standardisation pipeline as Part B
df_afrodb['mol'] = df_afrodb['SMILES'].apply(smiles_to_mol)
df_afrodb = df_afrodb[df_afrodb['mol'].notna()].copy().reset_index(drop=True)
print(f"AfroDb: {len(df_afrodb)} compounds parsed successfully")

# Compute ECFP4 fingerprints (same radius and nbits as Part D)
afrodb_fp_matrix = np.vstack([
    mol_to_fp(m, FP_RADIUS, FP_NBITS)
    for m in tqdm(df_afrodb['mol'], desc='AfroDb fingerprints')
])

# ============================================================
# TODO EXT4 — Project AfroDb fingerprints onto the existing UMAP.
# Use reducer.transform() — NOT fit_transform().
#
# Why not fit_transform?
#   fit_transform() would refit the UMAP to AfroDb data,
#   creating a completely different coordinate system.
#   transform() keeps the Malaria Box coordinate frame intact
#   so we can directly compare WHERE AfroDb compounds land.
#
# Fill in the blank:
#   afrodb_umap = reducer.___(afrodb_fp_matrix)
# ============================================================
afrodb_umap = reducer.___(afrodb_fp_matrix)   # TODO EXT4

# --- PRE-WRITTEN: Plot overlay ---
fig, ax = plt.subplots(figsize=(11, 8))

# MMV Malaria Box — background (grey)
ax.scatter(
    df_clean['UMAP_1'], df_clean['UMAP_2'],
    c='#BDBDBD', s=25, alpha=0.6, edgecolors='none', zorder=1,
    label=f'MMV Malaria Box (n={len(df_clean)})',
)

# AfroDb — foreground (green)
ax.scatter(
    afrodb_umap[:, 0], afrodb_umap[:, 1],
    c='#2E7D32', s=40, alpha=0.75, edgecolors='white', linewidths=0.3,
    marker='^', zorder=2,
    label=f'AfroDb African Natural Products (n={len(df_afrodb)})',
)

ax.set_xlabel('UMAP 1', fontsize=12)
ax.set_ylabel('UMAP 2', fontsize=12)
ax.set_title(
    'African Natural Products Projected onto Malaria Box Chemical Space\n'
    '(UMAP fitted on MMV only → AfroDb projected via reducer.transform())',
    fontsize=12,
)
ax.legend(fontsize=10, loc='upper right')

plt.tight_layout()
plt.show()

RDLogger.EnableLog('rdApp.*')

print()
print("💬 Discussion:")
print("  • Triangles inside or near grey MMV clusters → structurally similar to known anti-malarials")
print("  • Isolated triangles far from any grey cluster → novel chemical space — unexplored region")
print("  • What diseases could molecules in that unexplored region address?")

---
## 🎓 Session Summary

| Step | What we did | Tool | Key output |
|---|---|---|---|
| A | Environment setup | conda + pip | All libraries loaded |
| B | Data loading, SMILES parsing, salt removal, largest-fragment selection | pandas + RDKit MolStandardize | Clean `df_clean` with `mol` column |
| C | Lipinski descriptors + Ro5 flag | RDKit Descriptors | Descriptor columns in `df_clean` |
| D | Morgan fingerprints (ECFP4) | RDKit AllChem | `fp_matrix` (N × 2048) |
| E | Tanimoto similarity heatmap | RDKit DataStructs | Block-structured heatmap |
| F | PCA (linear) | scikit-learn | `PCA_1`, `PCA_2` coordinates |
| G | UMAP (non-linear, jaccard) | umap-learn | `UMAP_1`, `UMAP_2` coordinates |
| H | Property colouring (pIC50, LogP, Ro5) | matplotlib | Chemical space property maps |
| I1 | Capstone NP vs. MMV overlay | matplotlib | `chemical_space_map.png` |
| I2 | Scaffold diversity ratio + mean NN-Tanimoto | RDKit MurckoScaffold + DataStructs | Quantitative diversity comparison |
| Ext 1 | Murcko scaffold extraction + UMAP colouring | RDKit MurckoScaffold | Scaffold diversity ratio |
| Ext 2 | Interactive molecule browser | mols2grid | Paginated grid with SMARTS filter |
| Ext 3 | Activity cliff detection + structure drawing | RDKit DataStructs + Draw | Top-5 cliff pairs |
| Ext 4 | AfroDb overlay on Malaria Box UMAP | umap `reducer.transform()` | Out-of-sample NP projection |

---

### ⚡ Extension Notebooks

| Notebook | What | Sections |
|---|---|---|
| `functional_group_profiling.ipynb` | **Interpretable fingerprinting** — MACCS keys (J1) · Ertl fragments + χ² enrichment (J2) · UMAP panels by fragment (J3) · Five diversity metrics (J5) · Butina clustering + activity cliff analysis (J4) · MACCS vs ECFP4 UMAP (J6) | J1 · J2 (TODO) · J3 (TODO) · J5 (TODO) · J4 (TODO) · J6 (TODO) |

---

### Datasets used in this workshop

| Dataset | Role | URL |
|---|---|---|
| MMV Malaria Box (400 compounds + EC50) | Main dataset — Parts B–I, Ext 1–2 | [Direct XLS download](https://www.mmv.org/sites/default/files/uploads/docs/RandD/MalariaBox400compoundsDec2014.xls) |
| AfroDb Dataset S1 (954 natural products) | Overlay — Parts I, Ext 4 | [DOI: 10.1371/journal.pone.0078085](https://doi.org/10.1371/journal.pone.0078085) |
| PubChem AID 2302 (2 000 compounds, binary labels) | Activity cliff detection — Ext 3 | [pubchem.ncbi.nlm.nih.gov/bioassay/2302](https://pubchem.ncbi.nlm.nih.gov/bioassay/2302) |

---

### What's next?

- **`functional_group_profiling.ipynb`** *(Extension J)* — MACCS keys, Ertl fragment SAR, Butina clustering, diversity metrics, MACCS vs ECFP4
- **Scaffold Hunter** — hierarchical scaffold tree visualization: https://www.scaffoldhunter.com/
- **DataWarrior** — free GUI for chemical space exploration: https://openmolecules.org/datawarrior/
- **SwissADME** — online ADME/Lipinski predictions: http://www.swissadme.ch/
- **COCONUT** — natural product chemical space browser: https://coconut.naturalproducts.net/
- **NPASS v3** — natural products + bioactivity: https://bidd.group/NPASS/
- **ChEMBL-NTD** — pre-curated NTD bioactivity datasets: https://chembl.gitbook.io/chembl-ntd/downloads

---

> *"The purpose of exploring chemical space is not to make pretty plots —*  
> *it is to identify where the unexplored opportunities are,*  
> *and to ask: what diseases could be treated by molecules no one has made yet?"*  
> 
> *For African researchers, that question has an urgent and personal answer.*
